In [ ]:
# df_muestra

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
#Cargamos el modelo
import pickle
filename = 'modelo_xgb.pkl'
modelo, columnas = pickle.load(open(filename, 'rb'))
modelo

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
import pandas as pd

# 1. Leemos la muestra
df_muestra = pd.read_csv('data/df_muestra.csv')

data = df_muestra.copy()




In [ ]:
print(data.columns.tolist())

['customer_state', 'total_payment_value', 'total_freight', 'total_weight_g', 'total_volume_cm3', 'principal_category_by_price', 'principal_seller_state_by_price', 'purchase_year', 'facturacion_promedio_t1', 'facturacion_mediana_t1', 'facturacion_std_t1', 'facturacion_total_7d_avg', 'dias_entrega_promedio_7d_avg', 'dias_entrega_promedio_t2', 'dias_entrega_promedio_prev_month', 'es_post_domingo_festivo', 'es_diciembre', 'es_fin_de_semana_extendido', 'distancia_km', 'distancia_total_t1', 'distancia_total_prev_month', 'delivery_days']


In [24]:
# 2. Aplicamos el mismo get_dummies que usaste en el entrenamiento
data_encoded = pd.get_dummies(
    data,
    columns=['customer_state', 'principal_category_by_price', 'principal_seller_state_by_price']
)

# Alineamos con las columnas del modelo:
# - agrega con 0 las columnas dummy que no aparecieron en la muestra (categorías ausentes)
# - descarta cualquier columna que sobre y no sea parte de las 'columnas' del modelo


objetivo = 'delivery_days'
data_encoded = data.drop(objetivo, axis=1)
X_data = data_encoded.reindex(columns=columnas, fill_value=0)

Y_data = data['delivery_days']

# 3. Predicciones
predicciones = modelo.predict(X_data)

# 4. Comparamos real vs predicción
resultado = pd.DataFrame({
    'Real': Y_data,
    'Predicción': predicciones
})
resultado['Error'] = resultado['Real'] - resultado['Predicción']

print(resultado.tail(15))

      Real  Predicción     Error
1168   5.0    8.227263 -3.227263
1169  16.0   10.147776  5.852224
1170   5.0    5.890865 -0.890865
1171  18.0   12.303588  5.696412
1172   5.0    6.011114 -1.011114
1173   7.0    7.053315 -0.053315
1174  12.0    4.598622  7.401378
1175  20.0   10.041272  9.958728
1176   3.0    5.941052 -2.941052
1177   3.0    3.640125 -0.640125
1178   1.0    3.476324 -2.476324
1179   7.0    8.870741 -1.870741
1180   4.0    4.412218 -0.412218
1181  14.0    4.415515  9.584485
1182   2.0    4.804431 -2.804431


In [25]:
# Sumamos 3 días a la predicción
resultado['Prediccion_mas_3'] = resultado['Predicción'] + 3

# Comparamos si el valor real es inferior o igual a la predicción + 3
resultado['Cumple'] = np.where(resultado['Real'] <= resultado['Prediccion_mas_3'], 'Sí', 'No')

print(resultado.head(10))

   Real  Predicción     Error  Prediccion_mas_3 Cumple
0  11.0   12.297724 -1.297724         15.297724     Sí
1   5.0    5.098245 -0.098245          8.098245     Sí
2  12.0    7.877979  4.122021         10.877979     No
3  12.0   10.463483  1.536517         13.463483     Sí
4   9.0   10.035335 -1.035335         13.035335     Sí
5   1.0    4.156650 -3.156650          7.156650     Sí
6  11.0    9.567045  1.432955         12.567045     Sí
7  11.0    9.254434  1.745566         12.254434     Sí
8   5.0    6.141147 -1.141147          9.141147     Sí
9   7.0    6.254998  0.745002          9.254997     Sí


In [26]:
porcentaje_cumple = (resultado['Cumple'] == 'Sí').mean() * 100
print(f"Porcentaje de casos donde el real es <= predicción + 3: {porcentaje_cumple:.2f}%")

Porcentaje de casos donde el real es <= predicción + 3: 80.30%


# Informacion de tu pedido:

In [ ]:
# 2. Aplicamos el mismo get_dummies que usaste en el entrenamiento
data_encoded = pd.get_dummies(
    data,
    columns=['customer_state', 'principal_category_by_price', 'principal_seller_state_by_price']
)
data['prediccion'] = modelo.predict(data_encoded.reindex(columns=columnas, fill_value=0))

In [ ]:
print(df_muestra['customer_state'].unique())
print(df_muestra['principal_category_by_price'].unique())
print(df_muestra['principal_seller_state_by_price'].unique())

['RJ' 'SP']
["b'brinquedos'" "b'moveis_decoracao'" "b'cama_mesa_banho'"
 "b'beleza_saude'" "b'relogios_presentes'" "b'esporte_lazer'"
 "b'informatica_acessorios'" "b'utilidades_domesticas'"]
['SP' 'RJ' 'PR' 'MG']


In [28]:
print(df_muestra[['purchase_year']].describe())
print(df_muestra.columns.tolist())  # ya lo tengo, pero confirmemos si hay alguna columna de fecha completa (día/mes) que no habíamos visto

       purchase_year
count    1183.000000
mean     2017.552832
std         0.497411
min      2017.000000
25%      2017.000000
50%      2018.000000
75%      2018.000000
max      2018.000000
['customer_state', 'total_payment_value', 'total_freight', 'total_weight_g', 'total_volume_cm3', 'principal_category_by_price', 'principal_seller_state_by_price', 'purchase_year', 'facturacion_promedio_t1', 'facturacion_mediana_t1', 'facturacion_std_t1', 'facturacion_total_7d_avg', 'dias_entrega_promedio_7d_avg', 'dias_entrega_promedio_t2', 'dias_entrega_promedio_prev_month', 'es_post_domingo_festivo', 'es_diciembre', 'es_fin_de_semana_extendido', 'distancia_km', 'distancia_total_t1', 'distancia_total_prev_month', 'delivery_days']


In [30]:
%pip install streamlit

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/10.1 MB 10.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.4/10.1 MB 20.3 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 19.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/797.2 kB ? eta -:--:--
   --------------------------------------- 797.2/797.2 kB 29.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   --------------------- ------------------ 14.7/27.9 MB 70.9 MB/s eta 0:00:01
   ---------------------------------------  27.8/27.9 MB 68.1 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 62.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ---------------------------------------- 11.4/11.4 MB 55.9 MB/s eta 0:00:00
Note: you may need


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import streamlit as st
import pandas as pd
import pickle

# ---------------------------
# Cargar modelo y columnas
# ---------------------------
@st.cache_resource
def cargar_modelo():
    with open('modelo_xgb.pkl', 'rb') as archivo:
        modelo, columnas = pickle.load(archivo)
    return modelo, columnas

modelo, columnas = cargar_modelo()

# ---------------------------
# Cargar muestra de referencia
# ---------------------------
@st.cache_data
def cargar_muestra():
    return pd.read_csv('data/df_muestra.csv')

df_muestra = cargar_muestra()

st.title("Predicción de días de entrega")

# ---------------------------
# 1. Elegir caso de referencia (para variables históricas)
# ---------------------------
st.subheader("1. Caso de referencia (contexto histórico)")
indice_referencia = st.selectbox("Selecciona un registro de referencia", df_muestra.index)
caso_referencia = df_muestra.loc[indice_referencia]

columnas_historicas = [
    'facturacion_promedio_t1', 'facturacion_mediana_t1', 'facturacion_std_t1',
    'facturacion_total_7d_avg', 'dias_entrega_promedio_7d_avg',
    'dias_entrega_promedio_t2', 'dias_entrega_promedio_prev_month',
    'distancia_total_t1', 'distancia_total_prev_month'
]

with st.expander("Ver valores históricos usados"):
    st.write(caso_referencia[columnas_historicas])

# ---------------------------
# 2. Variables del pedido (editables)
# ---------------------------
st.subheader("2. Datos del pedido")

customer_state = st.selectbox("Estado del cliente", ['RJ', 'SP'])

principal_category_by_price = st.selectbox(
    "Categoría principal",
    ["b'brinquedos'", "b'moveis_decoracao'", "b'cama_mesa_banho'", "b'beleza_saude'",
     "b'relogios_presentes'", "b'esporte_lazer'", "b'informatica_acessorios'", "b'utilidades_domesticas'"]
)

principal_seller_state_by_price = st.selectbox("Estado principal del vendedor", ['SP', 'RJ', 'PR', 'MG'])

total_payment_value = st.number_input("Valor total del pago", min_value=0.0, value=100.0)
total_freight = st.number_input("Flete total", min_value=0.0, value=20.0)
total_weight_g = st.number_input("Peso total (g)", min_value=0.0, value=1000.0)
total_volume_cm3 = st.number_input("Volumen total (cm3)", min_value=0.0, value=5000.0)
distancia_km = st.number_input("Distancia (km)", min_value=0.0, value=500.0)
purchase_year = st.selectbox("Año de compra", [2017, 2018])

es_post_domingo_festivo = st.checkbox("¿Es post domingo/festivo?")
es_diciembre = st.checkbox("¿Es diciembre?")
es_fin_de_semana_extendido = st.checkbox("¿Es fin de semana extendido?")

# ---------------------------
# 3. Armar el registro completo
# ---------------------------
if st.button("Predecir"):
    entrada = {
        'customer_state': customer_state,
        'total_payment_value': total_payment_value,
        'total_freight': total_freight,
        'total_weight_g': total_weight_g,
        'total_volume_cm3': total_volume_cm3,
        'principal_category_by_price': principal_category_by_price,
        'principal_seller_state_by_price': principal_seller_state_by_price,
        'purchase_year': purchase_year,
        'facturacion_promedio_t1': caso_referencia['facturacion_promedio_t1'],
        'facturacion_mediana_t1': caso_referencia['facturacion_mediana_t1'],
        'facturacion_std_t1': caso_referencia['facturacion_std_t1'],
        'facturacion_total_7d_avg': caso_referencia['facturacion_total_7d_avg'],
        'dias_entrega_promedio_7d_avg': caso_referencia['dias_entrega_promedio_7d_avg'],
        'dias_entrega_promedio_t2': caso_referencia['dias_entrega_promedio_t2'],
        'dias_entrega_promedio_prev_month': caso_referencia['dias_entrega_promedio_prev_month'],
        'es_post_domingo_festivo': int(es_post_domingo_festivo),
        'es_diciembre': int(es_diciembre),
        'es_fin_de_semana_extendido': int(es_fin_de_semana_extendido),
        'distancia_km': distancia_km,
        'distancia_total_t1': caso_referencia['distancia_total_t1'],
        'distancia_total_prev_month': caso_referencia['distancia_total_prev_month'],
    }

    df_entrada = pd.DataFrame([entrada])

    # Aplicamos el mismo encoding que en el entrenamiento
    df_entrada_encoded = pd.get_dummies(
        df_entrada,
        columns=['customer_state', 'principal_category_by_price', 'principal_seller_state_by_price']
    )

    # Alineamos con las columnas del modelo
    X_input = df_entrada_encoded.reindex(columns=columnas, fill_value=0)

    prediccion = modelo.predict(X_input)[0]
    prediccion_mas_3 = prediccion + 3

    st.success(f"Predicción de días de entrega: {prediccion:.2f}")
    st.info(f"Predicción + margen (3 días): {prediccion_mas_3:.2f}")

    with st.expander("Ver datos enviados al modelo"):
        st.dataframe(X_input)

2026-09-21 21:01:14.414 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-21 21:01:14.422 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-21 21:01:14.424 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-21 21:01:14.425 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-21 21:01:14.434 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-21 21:01:14.754 
  command:

    streamlit run C:\Users\USUARIO\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-21 21:01:14.755 Th